In [1]:
import os
import json
import time
import pandas as pd
import sys
import os
import numpy as np
from datetime import date

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

import src.modify_reps
import src.gen_committees
import src.add_bioguide

In [3]:
try: 
    df = pd.read_json(os.path.join(project_root, "src", "generated_outputs", "congressmen_mod.json"))
except Exception as e:
    print("There is an issue with the congressmen.json. Quitting.")
    sys.exit()


In [4]:
df

,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,startYear,endYear,...,terms,valid_roles,education,military,illegal,work_history,gov_highlights,accolades,personal,uncaptured
0,M001244,Ashley Moody,Republican,Florida,https://api.congress.gov/v3/member/M001244?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/https://bi...,Senate,2025,2031,...,"[2025-01-21, 2029-01-03]",[],"[High school graduate, B.S., accounting, Unive...",[],[],"[lawyer, adjunct professor, assistant United S...",[],[],[],"[Hillsborough County, FL, circuit court judge,..."
1,W000812,Ann Wagner,Republican,Missouri,https://api.congress.gov/v3/member/W000812?for...,Image courtesy of the Member,https://www.congress.gov/img/member/695fc654dd...,House of Representatives,2013,2027,...,"[2013-01-03, 2027-01-03]",[],"[High school graduate, B.S.B.A., University of...",[],[],"[businesswoman, United States Ambassador to Lu...",[],[],[],"[chair, Missouri Republican Party, 1999-2005, ..."
2,R000619,Michael A. Rulli,Republican,Ohio,https://api.congress.gov/v3/member/R000619?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69401dcc8c...,House of Representatives,2024,2028,...,"[2024-06-11, 2027-01-03]",[],"[High school graduate, Emerson College, 1991]",[],[],"[business executive, member of the Ohio state ...",[member of the Leetonia Exempted Village Schoo...,[],[],[]
3,J000312,James C. Justice,Republican,West Virginia,https://api.congress.gov/v3/member/J000312?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/67c86b5e61...,Senate,2025,2031,...,"[2024-01-03, 2031-01-03]",[],"[High school graduate, B.A., Marshall Universi...",[],[],"[company founder and chief executive officer, ...",[agriculture and coal mining businessman],[],[],[]
4,D000628,Neal P. Dunn,Republican,Florida,https://api.congress.gov/v3/member/D000628?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/115_rp_fl_...,House of Representatives,2017,2027,...,"[2017-01-03, 2027-01-03]",[],"[B.S., Washington & Lee University, 1975, M.D....","[United States Army, 1989-2010]",[],"[urologist, banker]",[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532,G000546,Sam Graves,Republican,Missouri,https://api.congress.gov/v3/member/G000546?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/g000546_20...,House of Representatives,2001,2027,...,"[2001-01-03, 2027-01-03]",[],"[High school graduate, B.S., University of Mis...",[],[],[member of the Missouri state house of represe...,"[chair, Committee on Small Business (One Hundr...",[],[],[]
533,M001143,Betty McCollum,Democrat,Minnesota,https://api.congress.gov/v3/member/M001143?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/116_rp_mn_...,House of Representatives,2001,2027,...,"[2001-01-03, 2027-01-03]",[],"[High school graduate, A.A., Inver Hills Commu...",[],[],[member of the Minnesota state house of repres...,[],[],[],[]
534,C000537,James E. Clyburn,Democrat,South Carolina,https://api.congress.gov/v3/member/C000537?for...,Image courtesy of the Member,https://www.congress.gov/img/member/c000537_20...,House of Representatives,1993,2027,...,"[1993-01-03, 2027-01-03]",[],"[High school graduate, B.A., South Carolina St...",[],[],"[teacher, newspaper publisher, staff, Governor...","[chair, House Democratic Caucus (One Hundred N...",[awarded the Presidential Medal of Freedom by ...,[],"[employment counselor, South Carolina state em..."
535,K000009,Marcy Kaptur,Democrat,Ohio,https://api.congress.gov/v3/member/K000009?for...,Image courtesy of the Member,https://www.congress.gov/img/member/k000009_20...,House of Representatives,1983,2027,...,"[1983-01-03, 2027-01-03]",[],"[High school graduate, B.A., University of Wis...",[],[],"[assistant director for urban affairs, domesti...",[],[],[],"[post-graduate studies, Massachusetts Institut..."


In [ ]:
df[df['terms'].isna()]


,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,startYear,endYear,...,illegal,work_history,gov_highlights,accolades,personal,uncaptured,terms_start_date,terms_end_date,bioguide_tenure,bg_duration
114,T000490,David J. Taylor,Republican,Ohio,https://api.congress.gov/v3/member/T000490?for...,Image courtesy of the Member,https://www.congress.gov/img/member/677460190b...,House of Representatives,2025,2027,...,[],"[business owner, elected as a Republican to th...",[],[],[],"[assistant prosecutor, Clermont County, Ohio]",NaT,NaT,NaT,NaT
536,H000874,Steny H. Hoyer,Democrat,Maryland,https://api.congress.gov/v3/member/H000874?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/116_rp_md_...,House of Representatives,1981,2027,...,[],"[lawyer, private practice, member of the Maryl...",[member of the Maryland state board for higher...,[],[],[],NaT,NaT,NaT,NaT


In [ ]:
df['terms_start_date'] = df['terms'].str[0]
df['terms_start_date'] = df['terms_start_date'].fillna(df['startYear'].astype(str)+"-01-03")
df['terms_start_date'] = pd.to_datetime(df['terms_start_date'])
df['terms_end_date'] = df['terms'].str[1]
df['terms_end_date'] = df['terms_end_date'].fillna(df['endYear'].astype(str)+"-01-03")
df['terms_end_date'] = pd.to_datetime(df['terms_end_date'])

# 1. Get today's date
today = pd.Timestamp.now()
effective_end_date = np.where(df['terms_end_date'] > today, today, df['terms_end_date'])
df['bg_duration'] = (pd.to_datetime(effective_end_date) - df['terms_start_date']).dt.days

df['bg_duration'] = df['bg_duration'].astype(int)
df['bg_duration'] = df['bg_duration'] / 365

#tenure_all_time is across everyone, and across all time
df['bg_tenure_rank_all_time']  = df['bg_duration'].rank(ascending=False, method='min').astype(int)
df['bg_tenure_rank_all_time_party'] = df.groupby('partyName')['bg_duration'].rank(ascending=False, method='min').astype(int)

#tenure_current is just for current members, if they're not current members will be nan
df['bg_tenure_rank_current'] = np.where(df['current_member']=="yes", df.groupby(['current_member', 'chamber'])['bg_duration'].rank(ascending=False,method='min'), np.nan)
df['bg_tenure_rank_current_party'] = np.where(df['current_member']=="yes", df.groupby(['current_member', 'chamber','partyName'])['bg_duration'].rank(ascending=False,method='min'), np.nan)
df['bg_tenure_rank_current_party'] = df['bg_tenure_rank_current_party'].astype(pd.Int64Dtype())

df['bg_tenure_rank_current'] = df['bg_tenure_rank_current'].astype(pd.Int64Dtype())

df['bg_tenure_rank_current_percentile'] = np.where(df['current_member']=="yes", df.groupby(['current_member','chamber'])['bg_duration'].rank(ascending=True,method='max'), np.nan)
df['bg_tenure_rank_current_percentile'] = round(df['bg_tenure_rank_current_percentile']/df['chamber_current_count']*100).astype(pd.Int64Dtype())

df['bg_endYear'] = df['terms_end_date'].dt.year.astype(str)
df.drop(columns=['terms_start_date', 'terms_end_date'], inplace=True)
print(df['bg_duration'].dtype)
df

float64


,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,startYear,endYear,...,uncaptured,bg_duration,bg_tenure_rank_all_time,bg_tenure_rank_all_time_party,bg_tenure_rank_current,bg_tenure_rank_current_party,bg_tenure_rank_current_percentile,bg_endYear,terms_start_date,terms_end_date
0,M001244,Ashley Moody,Republican,Florida,https://api.congress.gov/v3/member/M001244?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/https://bi...,Senate,2025,2031,...,"[Hillsborough County, FL, circuit court judge,...",1.002740,531,270,99,52,2,2029,2025-01-21,2029-01-03
1,W000812,Ann Wagner,Republican,Missouri,https://api.congress.gov/v3/member/W000812?for...,Image courtesy of the Member,https://www.congress.gov/img/member/695fc654dd...,House of Representatives,2013,2027,...,"[chair, Missouri Republican Party, 1999-2005, ...",13.060274,140,62,96,42,78,2027,2013-01-03,2027-01-03
2,R000619,Michael A. Rulli,Republican,Ohio,https://api.congress.gov/v3/member/R000619?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69401dcc8c...,House of Representatives,2024,2028,...,[],1.616438,464,237,368,186,16,2027,2024-06-11,2027-01-03
3,J000312,James C. Justice,Republican,West Virginia,https://api.congress.gov/v3/member/J000312?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/67c86b5e61...,Senate,2025,2031,...,[],2.054795,450,229,92,48,9,2031,2024-01-03,2031-01-03
4,D000628,Neal P. Dunn,Republican,Florida,https://api.congress.gov/v3/member/D000628?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/115_rp_fl_...,House of Representatives,2017,2027,...,[],9.057534,218,101,154,69,65,2027,2017-01-03,2027-01-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532,G000546,Sam Graves,Republican,Missouri,https://api.congress.gov/v3/member/G000546?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/g000546_20...,House of Representatives,2001,2027,...,[],25.068493,45,13,32,7,93,2027,2001-01-03,2027-01-03
533,M001143,Betty McCollum,Democrat,Minnesota,https://api.congress.gov/v3/member/M001143?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/116_rp_mn_...,House of Representatives,2001,2027,...,[],25.068493,45,32,32,26,93,2027,2001-01-03,2027-01-03
534,C000537,James E. Clyburn,Democrat,South Carolina,https://api.congress.gov/v3/member/C000537?for...,Image courtesy of the Member,https://www.congress.gov/img/member/c000537_20...,House of Representatives,1993,2027,...,"[employment counselor, South Carolina state em...",33.073973,16,11,12,10,97,2027,1993-01-03,2027-01-03
535,K000009,Marcy Kaptur,Democrat,Ohio,https://api.congress.gov/v3/member/K000009?for...,Image courtesy of the Member,https://www.congress.gov/img/member/k000009_20...,House of Representatives,1983,2027,...,"[post-graduate studies, Massachusetts Institut...",43.082192,5,3,4,2,99,2027,1983-01-03,2027-01-03
